In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Dense, Dropout, Flatten, BatchNormalization,
                                     Conv2D, MaxPooling2D)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (EarlyStopping, ReduceLROnPlateau, ModelCheckpoint)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import categorical_crossentropy
from tensorflow.keras.optimizers import Adam

# скачиваем данные и разделяем на набор для обучения и тестовый
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(x_train.shape, y_train.shape)

(60000, 28, 28) (60000,)


In [2]:
num_classes = 10
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)
input_shape = (28, 28, 1)

# преобразование векторных классов в бинарные матрицы
y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255
print('Размерность x_train:', x_train.shape)
print(x_train.shape[0], 'Размер train')
print(x_test.shape[0], 'Размер test')

Размерность x_train: (60000, 28, 28, 1)
60000 Размер train
10000 Размер test


In [ ]:
model = Sequential()
model.add(Conv2D(16, kernel_size=(2, 2),activation='relu',input_shape=input_shape))
model.add(Conv2D(32, kernel_size=(3, 3),activation='relu'))
model.add(Dropout(0.3))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, (5, 5), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.3))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(num_classes, activation='softmax'))

# Заменяем Adadelta на Adam для более быстрого и стабильного обучения
model.compile(loss=categorical_crossentropy,
              optimizer=Adam(),
              metrics=['accuracy'])

d:\Repositories\MGTU\TOBD\ЛР (9)\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
batch_size = 512
epochs = 20

# Data Augmentation - генератор изменённых изображений для лучшего обобщения
datagen = ImageDataGenerator(
    rotation_range=10,      # Повороты до 10 градусов
    width_shift_range=0.1,  # Сдвиг по ширине
    height_shift_range=0.1, # Сдвиг по высоте
    zoom_range=0.1          # Масштабирование
)
datagen.fit(x_train)

# Callbacks для управления процессом обучения
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('mnist.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

# Обучение модели с использованием аугментации данных
hist = model.fit(datagen.flow(x_train, y_train, batch_size=batch_size),
                 epochs=epochs,
                 validation_data=(x_test, y_test),
                 callbacks=callbacks,
                 verbose=1)

print("Модель успешно обучена")
print("Лучшая модель сохранена как mnist.keras")

Epoch 1/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.9878 - loss: 0.0405
Epoch 1: val_accuracy improved from None to 0.99490, saving model to mnist.keras

Epoch 1: finished saving model to mnist.keras
118/118 ━━━━━━━━━━━━━━━━━━━━ 10s 85ms/step - accuracy: 0.9888 - loss: 0.0373 - val_accuracy: 0.9949 - val_loss: 0.0147 - learning_rate: 0.0010
Epoch 2/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.9899 - loss: 0.0335
Epoch 2: val_accuracy improved from 0.99490 to 0.99500, saving model to mnist.keras

Epoch 2: finished saving model to mnist.keras
118/118 ━━━━━━━━━━━━━━━━━━━━ 10s 85ms/step - accuracy: 0.9901 - loss: 0.0329 - val_accuracy: 0.9950 - val_loss: 0.0144 - learning_rate: 0.0010
Epoch 3/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.9902 - loss: 0.0331
Epoch 3: val_accuracy improved from 0.99500 to 0.99620, saving model to mnist.keras

Epoch 3: finished saving model to mnist.keras
118/118 ━━━━━━━━━━━━━━━━━━━━ 10s 85ms/step - accuracy: 0.9904 -

In [7]:
score = model.evaluate(x_test, y_test, verbose=0)
print('Потери на тесте:', score[0])
print('Точность на тесте:', score[1])

Потери на тесте: 0.01273827999830246
Точность на тесте: 0.9958999752998352
